[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%207/L14_Making_It_Yours.ipynb)

# Making It Yours — Part 2: the layer that **learns**
### ISBA 2411 · Week 7 · Lecture 14

Last lecture we built a support copilot for **Cobalt** without a single hand-sorted ticket. It read the
inbox, searched it by meaning, routed it into eight buckets, and knew when to ask a human. It scored
**64%** on its first guess. It cost nothing.

Tonight we ask the question that always comes next in a real company:

> **We have our own data. What would it actually buy us, and what would it cost?**

There are seven rungs on that ladder. Lecture 13 was rungs 1 and 2. Week 8 is rung 3. Tonight is
rungs 4 through 7 — and the honest answer about where you should stop climbing.

**Second half is a competition.** You will get a fixed budget of labels and have to decide *which*
tickets to spend it on. Same budget for every team. The results are not close.

---
### How to use this notebook
Run every cell in order (`Shift + Enter`). Steps marked **YOUR TURN** are the ones you change.
Nothing here needs a GPU except Step 10, which says so.

---
## Setup

In [ ]:
%%capture
%pip install -q transformers sentence-transformers scikit-learn

#### ▶ STEP 1 &middot; Load both ticket sets

In [ ]:
# -------- STEP 1 · Load both ticket sets --------
import pandas as pd, numpy as np, textwrap, json, urllib.request
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
np.set_printoptions(precision=3, suppress=True)

# two separate sets, and the difference matters for the rest of the night
pool = pd.read_csv("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_pool.csv")   # 1,400 tickets we COULD pay to have sorted
test = pd.read_csv("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_test.csv")   # the 160 hand-written tickets from Lecture 13

CATS = sorted(test.category.unique())
print(f"pool  {len(pool):5} tickets   <- the inbox we can buy labels from")
print(f"test  {len(test):5} tickets   <- hand-written, held back, our only honest scoreboard")
print(f"\ncategories: {CATS}")

> ### Why two sets, and why it matters
> Every number tonight is measured on the **160 hand-written tickets** — the same ones you saw last
> lecture. Those are never trained on. The 1,400-ticket `pool` is where we go shopping for labels.
>
> This is not a technicality. If we scored ourselves on tickets we had trained on, every number would
> look wonderful and none of them would predict what happens on Monday. **The test set is the only
> thing standing between you and fooling yourself.**

---
# Part 1 · Why adapting is cheap at all

#### ▶ STEP 2 &middot; Where Lecture 13 left off

In [ ]:
# -------- STEP 2 · Where Lecture 13 left off --------
from transformers import pipeline
router = pipeline("zero-shot-classification",
                  model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0")

# the exact same descriptions we wrote in Lecture 13
CATEGORIES = {
 "login_access"   : "signing in, passwords, two-factor authentication or account access",
 "billing_plan"   : "an invoice, payment, refund, subscription plan or pricing question",
 "data_sync"      : "a database connector failing to refresh or import data",
 "integrations"   : "connecting to another product such as Slack, Salesforce, webhooks or the API",
 "performance"    : "the product being slow, timing out or hanging",
 "bug_ui"         : "a chart, table or screen displaying something incorrectly",
 "how_to"         : "asking for instructions on how to do something",
 "feature_request": "requesting a new capability that does not exist yet",
}
desc2cat = {v: k for k, v in CATEGORIES.items()}

out = router(test.ticket_text.tolist(), candidate_labels=list(CATEGORIES.values()),
             multi_label=False, batch_size=8)
zs_pred = [desc2cat[o["labels"][0]] for o in out]
zs_conf = np.array([o["scores"][0] for o in out])

ZS_ACC = accuracy_score(test.category, zs_pred)
ZS_F1  = f1_score(test.category, zs_pred, average="macro")
print(f"Lecture 13's copilot, no training data at all:")
print(f"   accuracy  {ZS_ACC:.1%}")
print(f"   macro-F1  {ZS_F1:.3f}   <- our number to beat all night")

✅ **What just happened.** We reproduced last lecture's result exactly. This is the line every rung
above has to beat, and beat by enough to justify its cost.

**A word on macro-F1**, because we will use it all night instead of accuracy. Accuracy asks "what
share did we get right" and lets a model look good by nailing the big easy buckets. Macro-F1 scores
every category separately and then averages, so **a bucket you never get right cannot hide**. When you
are deciding where to spend money, that is the number that tells you the truth.

#### ▶ STEP 3 &middot; Find the buckets your data could fix

In [ ]:
# -------- STEP 3 · Find the buckets your data could fix --------
per_cat = (pd.DataFrame({"true": test.category, "pred": zs_pred})
           .assign(ok=lambda d: d.true == d.pred)
           .groupby("true").ok.mean().sort_values())

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(per_cat.index, per_cat.values,
        color=["#E11D48" if v < .5 else "#059669" for v in per_cat.values])
ax.axvline(.5, ls="--", c="#334155", lw=1); ax.set_xlim(0, 1)
ax.set_title("Where a general model is already good enough, and where it is not")
ax.set_xlabel("share of that bucket caught correctly")
plt.tight_layout(); plt.show()

print("Worst two buckets, i.e. where our own data has the most to offer:")
for c in per_cat.index[:2]:
    print(f"   {c:16} {per_cat[c]:.0%}")

✅ **What just happened.** The same picture from last lecture. Some buckets are already fine with no
data at all. Two are genuinely broken.

💼 **At work this means:** this chart is a **shopping list**. It says where your money would do
something and where it would be wasted. Nobody labels data because labelling is virtuous; you label
the part the free version cannot do.

---
## Why is any of this possible?

A fair question you should be asking. How can a model that has never seen a Cobalt ticket, never seen
our product, never been told what our categories are, get **64%** of them right?

Because somebody already paid to teach it English. Here is literally the exercise they used.

#### ▶ STEP 4 &middot; Fill in the blank

In [ ]:
# -------- STEP 4 · Fill in the blank --------
from transformers import pipeline
filler = pipeline("fill-mask", model="bert-base-uncased")

sentences = [
 "Our SSO through Okta stopped working and nobody can [MASK] in.",
 "We were charged [MASK] this month for the same seats.",
 "Exporting anything over ten thousand rows just times [MASK].",
 "Please add [MASK] mode.",
]
for s in sentences:
    top = filler(s, top_k=4)
    print(s)
    print("   ->", ",  ".join(f"{t['token_str']} ({t['score']:.0%})" for t in top), "\n")

✅ **What just happened.** That is **masked language modelling**, and it is the whole trick behind BERT.
Take an ordinary sentence, hide one word, make the model guess it. No human labelled anything — the
answer was already in the text. So you can run this on the entire internet essentially for free.

Do that a few hundred billion times and, to guess the missing word, the model is forced to learn
grammar, word meaning, and which words keep company with which. **Nobody set out to teach it that.
It is a side effect of playing fill-in-the-blank at enormous scale.**

💼 **At work this means:** when a vendor says a model was "pretrained", this is what they bought with
their millions. You are renting the result. Every rung on tonight's ladder is a different answer to
"how much do I add on top of what they already paid for?"

#### ▶ STEP 5 &middot; Fill in the blank on Cobalt jargon

In [ ]:
# -------- STEP 5 · Fill in the blank on Cobalt jargon --------
# now the same game, but on OUR vocabulary
jargon = "The [MASK] connector stopped syncing our warehouse data."
print(jargon, "\n")
for t in filler(jargon, top_k=8):
    print(f"   {t['token_str']:14} {t['score']:.1%}")

✅ **What just happened.** Ask it about *our* world and the guesses go generic. It knows a connector is
a thing that connects. It has no idea that Cobalt customers mean Salesforce, Snowflake or Okta,
because those words were rare in whatever it read.

**This is exactly the gap your data closes**, and it is why rung 6 ("keep pretraining it on your own
text") exists at all. Hold that thought — we will price it later.

#### ▶ STEP 6 &middot; Why BERT reads and GPT writes

In [ ]:
# -------- STEP 6 · Why BERT reads and GPT writes --------
# Encoder or decoder? Same transformer block, different direction of attention.
gen = pipeline("text-generation", model="gpt2")
prompt = "Our SSO through Okta stopped working and"
print("GPT-2 (a DECODER) continues the sentence:")
print("  ", gen(prompt, max_new_tokens=20, do_sample=False,
                pad_token_id=50256)[0]["generated_text"], "\n")

print("BERT (an ENCODER) fills a gap in the middle, using BOTH sides:")
print("  ", filler("Our SSO through Okta stopped [MASK] and nobody can log in.",
                   top_k=3)[0]["sequence"])

✅ **What just happened.** Two models, same building block from last lecture, one difference:

| | sees | good at | our use |
|---|---|---|---|
| **BERT** (encoder) | the words **before and after** | understanding a finished text | sorting tickets |
| **GPT** (decoder) | only the words **before** | producing the next word | drafting a reply |

That is the entire distinction, and it follows from one design choice: whether a word is allowed to
attend to words on its right. BERT can look ahead, so it reads well and cannot write. GPT cannot look
ahead, so it writes well and reads a little worse.

💼 **At work this means:** for classifying, tagging, routing or searching your own documents, a small
encoder is usually the right and much cheaper tool. Reaching for a large chat model to sort tickets is
paying for a faculty you are not using.

---
## Rung 4 and rung 5 · what your own labels buy

Both rungs mean "train on tickets a person has sorted". Before we spend any, look at what one
actually is.

#### ▶ STEP 7 &middot; What one labelled ticket costs

In [ ]:
# -------- STEP 7 · What one labelled ticket costs --------
sample = pool.sample(3, random_state=3)
for _, r in sample.iterrows():
    print(textwrap.fill(r.ticket_text, 96))
    print(f"   a person read that and wrote:  {r.category}\n")

print("That second line is the expensive part.")
print(f"At roughly 20 seconds per ticket, labelling all {len(pool):,} costs about "
      f"{len(pool)*20/3600:.1f} hours of somebody's attention.")
print("Add written guidelines and a second reader to check agreement and it is a week or two of work.")

#### ▶ STEP 8 &middot; Freeze the encoder, take the features

In [ ]:
# -------- STEP 8 · Freeze the encoder, take the features --------
# The encoder from Lecture 13, used as a FROZEN feature extractor. We are not changing it.
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_pool = encoder.encode(pool.ticket_text.tolist(), batch_size=64,
                        normalize_embeddings=True, show_progress_bar=True)
X_test = encoder.encode(test.ticket_text.tolist(), batch_size=64,
                        normalize_embeddings=True, show_progress_bar=False)
y_pool, y_test = pool.category.values, test.category.values
print(f"\nevery ticket is now {X_pool.shape[1]} numbers:  pool {X_pool.shape}, test {X_test.shape}")

✅ **What just happened.** Nothing was trained. We ran each ticket through last lecture's encoder once
and kept the vector. The encoder is **frozen** — its 23 million numbers will not change tonight.

The idea of rung 4 is that those vectors already separate the categories reasonably well, so all we
need to learn is a simple rule for drawing lines between them. That rule is tiny and cheap.

#### ▶ STEP 9 &middot; RUNG 4: train a head on 320 labels

In [ ]:
# -------- STEP 9 · RUNG 4: train a head on 320 labels --------
rng = np.random.default_rng(100)
idx320 = rng.choice(len(pool), 320, replace=False)

head = LogisticRegression(max_iter=2000, C=4.0).fit(X_pool[idx320], y_pool[idx320])
pred = head.predict(X_test)

print(f"RUNG 4  frozen encoder + a small head, trained on 320 labelled tickets")
print(f"   accuracy  {accuracy_score(y_test, pred):.1%}     (zero-shot was {ZS_ACC:.1%})")
print(f"   macro-F1  {f1_score(y_test, pred, average='macro'):.3f}   (zero-shot was {ZS_F1:.3f})")
print(f"\n   parameters we actually trained: {head.coef_.size:,}")
print(f"   parameters we left frozen:      ~22,700,000")

✅ **What just happened.** 320 sorted tickets — under two hours of somebody's afternoon — bought a
clear jump over the free version. And we trained a rounding error's worth of parameters to get it.

🔮 **Before you run the next cell, predict:** rung 5 fine-tunes the **whole** model instead of just the
head. Roughly 66 million parameters instead of three thousand. Same 320 tickets. Does it do better?

Write your guess down. Most people get this wrong.

#### ▶ STEP 10 &middot; RUNG 5: fine-tune the whole model

In [ ]:
# -------- STEP 10 · RUNG 5: fine-tune the whole model --------
# RUNG 5. The real thing: every weight in the model is allowed to move.
# CPU: about 3 minutes. GPU: about 20 seconds (Runtime > Change runtime type > T4).
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"training on {dev}")
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
mdl = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=len(CATS)).to(dev)
c2i = {c: i for i, c in enumerate(CATS)}

def batches(texts, labels, bs=16, shuffle=True):
    b = tok(list(texts), truncation=True, padding="max_length",
            max_length=64, return_tensors="pt")
    ds = TensorDataset(b["input_ids"], b["attention_mask"],
                       torch.tensor([c2i[l] for l in labels]))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

opt = torch.optim.AdamW(mdl.parameters(), lr=3e-5)
mdl.train()
for epoch in range(20):
    for ids, am, y in batches(pool.ticket_text.values[idx320], y_pool[idx320]):
        opt.zero_grad()
        mdl(input_ids=ids.to(dev), attention_mask=am.to(dev), labels=y.to(dev)).loss.backward()
        opt.step()

mdl.eval(); preds = []
for ids, am, _ in batches(test.ticket_text.values, y_test, bs=32, shuffle=False):
    with torch.no_grad():
        preds += mdl(input_ids=ids.to(dev),
                     attention_mask=am.to(dev)).logits.argmax(-1).cpu().tolist()
ft_pred = [CATS[i] for i in preds]

print(f"\nRUNG 5  full fine-tune, {sum(p.numel() for p in mdl.parameters()):,} parameters, same 320 tickets")
print(f"   accuracy  {accuracy_score(y_test, ft_pred):.1%}")
print(f"   macro-F1  {f1_score(y_test, ft_pred, average='macro'):.3f}")
print(f"\n   rung 4 (frozen + head) got macro-F1 {f1_score(y_test, head.predict(X_test), average='macro'):.3f}")

✅ **What just happened, and this is the most useful surprise of the night.** The far more expensive
option **lost**. Twenty thousand times more parameters, minutes instead of seconds of training, and it
scored worse.

Why? With only 320 examples there is not enough signal to responsibly move 66 million weights. The
model has more freedom than evidence, so it fits the noise in your sample. Freezing the encoder is a
way of saying *"stay as you are, you already know English — I will only learn the last step."* With
small data, that constraint is a feature.

💼 **At work this means:** "we fine-tuned a model" is not automatically better than "we trained a
classifier on embeddings", and it costs far more in time, money and things that can go wrong. **Ask
what the cheaper rung scores before you approve the expensive one.** Fine-tuning starts to pay off at
thousands of labels, not hundreds — which we can now show you.

#### ▶ STEP 11 &middot; The learning curve

In [ ]:
# -------- STEP 11 · The learning curve --------
curves = json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/l14_curves.json").read())
ns   = [int(k) for k in curves["frozen_head"]]
fh   = [curves["frozen_head"][str(n)]["f1"] for n in ns]
ft   = [curves["fine_tune"][str(n)]["f1"]  for n in ns]

fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.axhline(curves["zero_shot_f1"], ls="--", c="#0EA5E9", lw=1.8)
ax.text(ns[0], curves["zero_shot_f1"] + .012, "  rungs 1-2: no labels at all",
        color="#0EA5E9", fontsize=9.5, va="bottom")
ax.plot(ns, fh, "o-", color="#059669", lw=2.4, label="rung 4: frozen encoder + small head")
ax.plot(ns, ft, "s-", color="#7C3AED", lw=2.4, label="rung 5: fine-tune the whole model")
ax.set_xscale("log"); ax.set_xticks(ns); ax.set_xticklabels(ns)
ax.set_xlabel("tickets a person had to read and sort"); ax.set_ylabel("macro-F1 on held-out tickets")
ax.set_title("What your own labels actually buy"); ax.legend(frameon=False, loc="lower right")
ax.grid(alpha=.25); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print("Measured ahead of time with the code you just ran, 3-5 seeds per point.")

✅ **What just happened.** Three things, and each is a business decision rather than a technical one.

1. **The cheap rung is above the expensive one everywhere on this chart.** Not by a little. At every
   budget we can realistically afford, freezing the encoder wins.
2. **You need surprisingly few labels to beat the free version.** The green line crosses the dashed
   line early. A couple of hundred sorted tickets is a genuinely good afternoon's work.
3. **The curve bends.** Early labels are worth a lot; later ones are worth much less.

#### ▶ STEP 12 &middot; Find where the curve flattens

In [ ]:
# -------- STEP 12 · Find where the curve flattens --------
gain = [(ns[i], fh[i], (fh[i]-fh[i-1])/(ns[i]-ns[i-1])*100) for i in range(1, len(ns))]
print(f"{'labels':>8}  {'macro-F1':>9}   F1 points gained per 100 extra labels")
print(f"{ns[0]:>8}  {fh[0]:>9.3f}")
for n, f, g in gain:
    print(f"{n:>8}  {f:>9.3f}   {g*100:>6.2f}  {'#' * max(0, int(g*400))}")

print("\nThe question is never 'is more data better'. It is:")
print("does the NEXT hundred labels pay for itself?")

✅ **What just happened.** We turned the curve into the only question a manager actually has to answer.
The first hundred labels are transformative. Somewhere further along, another hundred buys a fraction
of a point, and that person's week is better spent elsewhere.

💼 **At work this means:** "we need more training data" is an incomplete sentence. Plot this curve for
*your* problem, find where it flattens, and stop there deliberately. Teams routinely spend months
labelling past the point where it changed any decision.

---
---
# Part 2 · The game: spend your labelling budget

You have just seen **how many** labels to buy. Now the harder question, and the one that separates
people who have actually done this from people who have only read about it:

## Which tickets do you pay to have labelled?

**The situation.** You are Cobalt's analytics lead. You have 1,400 unsorted tickets and budget for a
contractor to read and sort exactly **240** of them. You cannot see the answers before you choose —
that is the entire point, you are deciding what to *buy*.

**Teams.** Your **final project team**. Same team, same working relationship, no time lost forming
groups.

**The rules.**
1. Everyone gets the same 240-label budget and the same 1,400-ticket pool.
2. You may use the ticket text, the copilot's guess, its confidence, plan, region, priority, and the
   vectors in `X_pool`.
3. You may **not** use the true category to choose. That is what the budget buys.
4. Your score is **macro-F1 on the 160 held-out hand-written tickets**, averaged over 5 seeds.
5. To have beaten the house you must clear **random plus one standard deviation**, not just random.

**How the winner is picked.**

| | |
|---|---|
| **Primary** | highest official score (5-seed mean) |
| **Must clear** | the house bar. If no team clears it, no team beat the house, and that is a real result |
| **Ties** | anything within **0.010** is a tie, because that is inside the noise |
| **Tie-break** | the one-sentence explanation of *why* the rule should work |
| **Verification** | the leading team's function gets re-run on the projector |

That last row matters. The number alone is not the prize; a rule you can explain is. A team that
scores 0.72 and knows exactly why beats a team that scores 0.73 by accident.

#### ▶ STEP 13 &middot; THE GAME: your labelling budget

In [ ]:
# -------- STEP 13 · THE GAME: your labelling budget --------
BUDGET = 240
_ANSWERS = pool.category.values          # you are paying to see these

def buy_labels(chosen):
    chosen = np.asarray(list(chosen), dtype=int)
    if len(chosen) > BUDGET:
        raise ValueError(f"over budget: you picked {len(chosen)}, you can afford {BUDGET}")
    if len(set(chosen)) != len(chosen):
        raise ValueError("you paid twice for the same ticket")
    return chosen, _ANSWERS[chosen]

def score(chosen, name="unnamed", seeds=(0, 1, 2), show=True):
    chosen, y = buy_labels(chosen)
    if len(set(y)) < 2:
        raise ValueError("only one category in your selection, nothing to learn")
    f1s = []
    for s in seeds:
        clf = LogisticRegression(max_iter=2000, C=4.0, random_state=s).fit(X_pool[chosen], y)
        f1s.append(f1_score(y_test, clf.predict(X_test), average="macro"))
    f1 = float(np.mean(f1s))
    if show:
        n_cat = len(set(y))
        print(f"{name:26} {len(chosen):4} labels   macro-F1 {f1:.3f}   "
              f"({n_cat}/8 categories represented)")
    return f1

LEADERBOARD = {}
def post(name, f1):
    LEADERBOARD[name] = f1
    print(f"\n{'':3}{'TEAM':26}{'macro-F1':>10}")
    for i, (k, v) in enumerate(sorted(LEADERBOARD.items(), key=lambda x: -x[1]), 1):
        medal = ["1st", "2nd", "3rd"][i-1] if i <= 3 else f"{i}th"
        print(f"{medal:>3} {k:26}{v:>10.3f}")

def cluster_centres(X, k, seed=0):
    """Group the tickets into k clusters, return the row nearest the middle of each.

    One line of use, but this is the whole 'buy tickets unlike each other' idea:
    similar tickets land in the same cluster, so taking one per cluster spreads your
    purchase across every kind of ticket in the inbox.
    """
    from sklearn.cluster import KMeans
    km = KMeans(n_clusters=k, n_init=4, random_state=seed).fit(X)
    out = []
    for c in range(k):
        rows = np.where(km.labels_ == c)[0]
        if len(rows):
            out.append(rows[((X[rows] - km.cluster_centers_[c]) ** 2).sum(1).argmin()])
    return np.array(out)

def fill_to_budget(picks, budget=BUDGET, seed=0, n=None):
    """Housekeeping so you never have to think about it.

    Removes duplicates, tops up with random tickets if your rule returned too few,
    and trims if it returned too many. Call this at the end of your strategy.
    """
    n = len(pool) if n is None else n
    picks = np.array(sorted(set(np.asarray(list(picks), dtype=int))))
    if len(picks) < budget:
        spare = np.setdiff1d(np.arange(n), picks)
        extra = np.random.default_rng(seed).choice(spare, budget - len(picks), replace=False)
        picks = np.concatenate([picks, extra])
    return picks[:budget]

print(f"budget: {BUDGET} labels out of {len(pool)} tickets "
      f"({BUDGET/len(pool):.0%} of the inbox)")
print(f"to beat: zero-shot macro-F1 {ZS_F1:.3f}")

#### ▶ STEP 14 &middot; Look at the pool without the answers

In [ ]:
# -------- STEP 14 · Look at the pool without the answers --------
# This is all you get to see. Note what is missing.
pool_view = pool.drop(columns=["category"])
print("columns you may use:", list(pool_view.columns), "\n")
print(pool_view.head(4).to_string(max_colwidth=54))

assert "category" not in pool_view.columns, "no peeking"
print("\n'zs_guess' and 'zs_confidence' are last lecture's copilot run over the pool.")
print("They cost nothing -- no human sorted anything to produce them.")

> ### Why you are allowed the copilot's guess but not the answer
> This is not an artificial handicap, it is exactly the real situation. Running a zero-shot model over
> your whole inbox is free and takes an afternoon. Having a person read and sort a ticket costs money.
>
> So a real team genuinely does start with a free, unreliable guess plus a confidence score for every
> ticket, and a budget to buy a few real answers. **What you do with that is the job.**

#### ▶ STEP 15 &middot; The house baseline: random

In [ ]:
# -------- STEP 15 · The house baseline: random --------
# Random is a DISTRIBUTION, not one number. Comparing your rule against a single lucky
# draw would be unfair to you, or to it. So the house plays 12 hands.
draws = [score(np.random.default_rng(s).choice(len(pool), BUDGET, replace=False),
               f"random draw {s}", show=False) for s in range(12)]
HOUSE, SPREAD = float(np.mean(draws)), float(np.std(draws))

print(f"the house, random selection, 12 draws")
print(f"   mean macro-F1   {HOUSE:.3f}")
print(f"   spread          +/- {SPREAD:.3f}   (worst {min(draws):.3f}, best {max(draws):.3f})")
print(f"\n   zero-shot, no labels at all: {ZS_F1:.3f}")
print(f"\nSo the bar to genuinely clear is about {HOUSE + SPREAD:.3f}.")
print("Anything inside the spread is luck, not a better strategy.")
post("the house: random", HOUSE)

---
# ▶ YOUR TURN

## What the code has to do

One job: **return 240 row numbers** from the 1,400 in `pool_view`. Those are the tickets you are
paying a person to read and sort. Nothing else.

In the next cell there are **six options, A to F.** Exactly one is switched on. To try a different
one, you **delete the `#` at the start of its `picks = ...` line and add a `#` to the one that was
already on.** That is the whole mechanic. You do not have to write a function from scratch.

## What you have to work with

Everything below costs nothing and is allowed. The true category is not on this list, because that
is what your 240 labels are buying.

| you can use | what is in it | useful because |
|---|---|---|
| `view.zs_confidence` | a number from **0.179 to 0.996**, median 0.751 | how sure last lecture's copilot was |
| `view.zs_guess` | one of the **8** category names | the copilot's guess. Right about 64% of the time |
| `view.ticket_text` | the words, **32 to 224** characters long | length, or search for a word |
| `view.plan` | Free, Team, Business, Enterprise | 4 groups to spread across |
| `view.region` | North America, EMEA, APAC, LATAM | 4 groups to spread across |
| `view.priority` | low, normal, high, urgent | 4 groups to spread across |
| `view.channel` | email, in-app, chat, phone | 4 groups to spread across |
| `X_pool` | 1,400 rows of 384 numbers, one row per ticket | the vectors from Lecture 13. Similar tickets have similar rows |

## The four kinds of move you can make

Every sensible strategy is one of these, or a combination:

1. **Rank** — sort every ticket by one number and take the top or bottom 240. *(Options B and C)*
2. **Stratify** — split the budget evenly across the values of one column. *(Options A and D)*
3. **Threshold** — keep only the tickets above or below some cut, then sample from those.
4. **Cover** — use `X_pool` to buy tickets that are as different from each other as possible.
   *(Option E)*

One of these four does much better than the other three. Decide which one you believe in and why
**before** you run it.

## One thing worth noticing before you choose

The copilot's guesses are very lopsided. It guessed `bug_ui` **403** times and `data_sync` only
**28** times, out of 1,400. So a rule that asks for 30 tickets from each guessed category cannot get
30 `data_sync` tickets, because there are only 28 to be had. And `data_sync` is one of the two
categories that most needed help.

Whatever rule you pick decides which categories you end up owning labels for. That is worth thinking
about for a minute before you write anything.

#### ▶ STEP 16 &middot; YOUR TURN: write your strategy

In [ ]:
# -------- STEP 16 · YOUR TURN: write your strategy --------
def my_strategy(view, budget=BUDGET, seed=0):
    rng = np.random.default_rng(seed)

    # =========================================================================
    # SWITCH ON EXACTLY ONE OPTION.
    # To turn an option ON  : delete the "#" in front of its "picks =" lines.
    # To turn an option OFF : put a "#" back in front of them.
    # Right now OPTION A is on and B to F are off.
    # =========================================================================

    # --- OPTION A -- an even share from each of the copilot's 8 guessed categories ------
    per = budget // 8
    picks = np.concatenate([rng.choice(np.where(view.zs_guess.values == c)[0],
                                       min(per, (view.zs_guess.values == c).sum()),
                                       replace=False)
                            for c in sorted(view.zs_guess.unique())])

    # --- OPTION B -- the tickets the copilot was LEAST sure about ----------------------
    # picks = np.argsort(view.zs_confidence.values)[:budget]

    # --- OPTION C -- the tickets the copilot was MOST sure about -----------------------
    # picks = np.argsort(view.zs_confidence.values)[::-1][:budget]

    # --- OPTION D -- an even share from each value of one column you choose ------------
    #                 change COLUMN to "plan", "region", "priority" or "channel"
    # COLUMN = "priority"
    # groups = sorted(view[COLUMN].unique())
    # per    = budget // len(groups)
    # picks  = np.concatenate([rng.choice(np.where(view[COLUMN].values == g)[0],
    #                                    min(per, (view[COLUMN].values == g).sum()),
    #                                    replace=False)
    #                          for g in groups])

    # --- OPTION E -- buy tickets that are as DIFFERENT from each other as possible -----
    #                 groups the 1,400 tickets into 240 clusters of similar tickets,
    #                 then buys one ticket from the middle of each cluster
    # picks = cluster_centres(X_pool, budget, seed)

    # --- OPTION F -- only tickets the copilot was unsure about, then spread them out ---
    #                 a THRESHOLD move. Change 0.6 to move the cut.
    # unsure = np.where(view.zs_confidence.values < 0.6)[0]
    # print(f"{len(unsure)} tickets are below that confidence cut")
    # picks  = rng.choice(unsure, min(budget, len(unsure)), replace=False)

    # =========================================================================
    # Leave this line alone. It removes duplicates and tops up to exactly 240.
    return fill_to_budget(picks, budget, seed, len(view))


mine = my_strategy(pool_view)
print(f"picked {len(mine)} tickets")
print(f"\nthe copilot's guesses among the tickets you chose:")
print("  " + pool_view.zs_guess.iloc[mine].value_counts().to_string().replace("\n", "\n  "))
print("\nThat is not the true answer, but a very lopsided list here is a warning sign.")

### Scoring, so that it is fair

Your **official score** is your rule run on **five different random seeds** and averaged.

That is not bureaucracy, it is the same point we made about the house. A single run of anything
here moves by a couple of hundredths on luck alone. If everyone reported their best run out of five,
the leaderboard would be a ranking of luck. So `submit()` runs all five and averages, and it also
reports the spread, which tells you whether your rule is *reliable* or just *occasionally lucky*.

**On peeking:** the answers are sitting in `pool.category` and nothing stops you looking. Do not
bother. Balancing perfectly on the true labels scores about **0.706**, which loses to the best honest
strategy and is barely better than random. Cheating at this game does not even work, which is itself
worth knowing.

#### ▶ STEP 17 &middot; Score it and post to the leaderboard

In [ ]:
# -------- STEP 17 · Score it and post to the leaderboard --------
TEAM_NAME = "Team 1"          # <- your final project team

def submit(fn, name, seeds=range(5)):
    """Official scoring: run the rule on 5 seeds, average, and check the rules."""
    runs = []
    for s in seeds:
        idx = np.asarray(list(fn(pool_view, BUDGET, s)), dtype=int)
        if len(idx) > BUDGET:      raise ValueError(f"over budget: {len(idx)} > {BUDGET}")
        if len(set(idx)) != len(idx): raise ValueError("the same ticket was bought twice")
        runs.append(score(idx, name, show=False))
    mean, sd = float(np.mean(runs)), float(np.std(runs))
    bar = HOUSE + SPREAD
    print(f"  {name}")
    print(f"    official score   {mean:.3f}      <- this is what goes on the board")
    print(f"    spread           +/- {sd:.3f}   (over {len(runs)} seeds)")
    print(f"    the house        {HOUSE:.3f}")
    print(f"    bar to clear     {bar:.3f}      (house + one spread)")
    print(f"    result           {'BEAT THE HOUSE' if mean > bar else 'did not clear the bar'}")
    return mean

MY_F1 = submit(my_strategy, TEAM_NAME)
post(TEAM_NAME, MY_F1)

print("\nWrite two things on the board: your official score, and one sentence")
print("saying WHY your rule should work. The sentence is part of the judging.")

---
## The reveal

Five strategies, the same 240-label budget, all scored the same way.

#### ▶ STEP 18 &middot; THE REVEAL: six strategies, same budget

In [ ]:
# -------- STEP 18 · THE REVEAL: six strategies, same budget --------
from sklearn.cluster import KMeans

def strat_random(view, budget=BUDGET, seed=0):
    return np.random.default_rng(seed).choice(len(view), budget, replace=False)

def strat_balanced_guess(view, budget=BUDGET, seed=0):
    # even split across the copilot's GUESS. Legal, but the guess is only 64% right.
    rng = np.random.default_rng(seed); per = budget // len(CATS); picks = []
    for c in CATS:
        rows = np.where(view.zs_guess.values == c)[0]
        picks.append(rng.choice(rows, min(per, len(rows)), replace=False))
    picks = np.concatenate(picks)
    if len(picks) < budget:
        spare = np.setdiff1d(np.arange(len(view)), picks)
        picks = np.concatenate([picks, rng.choice(spare, budget - len(picks), replace=False)])
    return picks[:budget]

def strat_least_conf(view, budget=BUDGET, seed=0):
    return np.argsort(view.zs_confidence.values)[:budget]

def strat_most_conf(view, budget=BUDGET, seed=0):
    return np.argsort(view.zs_confidence.values)[::-1][:budget]

def strat_weak_first(view, budget=BUDGET, seed=0):
    rng = np.random.default_rng(seed)
    weak = list(per_cat.index[:3])
    rows = np.where(view.zs_guess.isin(weak).values)[0]
    w = rng.choice(rows, min(budget // 2, len(rows)), replace=False)
    rest = np.setdiff1d(np.arange(len(view)), w)
    return np.concatenate([w, rng.choice(rest, budget - len(w), replace=False)])

def strat_cover_space(view, budget=BUDGET, seed=0):
    # Exactly what OPTION E in Step 16 does: cluster the vectors, buy one ticket from
    # the middle of each cluster. Uses no labels at all, so it is completely legal.
    picks = np.array(sorted(set(cluster_centres(X_pool, budget, seed))))
    if len(picks) < budget:
        spare = np.setdiff1d(np.arange(len(view)), picks)
        rng = np.random.default_rng(seed)
        picks = np.concatenate([picks, rng.choice(spare, budget - len(picks), replace=False)])
    return picks[:budget]

# Every strategy that involves a random choice gets averaged over 5 selections, for
# exactly the reason we averaged the house over 12 draws: one lucky pick is not a result.
# least/most confident are deterministic, so one run is all there is.
STRATS = [("cover the space",       strat_cover_space,    5),
          ("random (the house)",    strat_random,         5),
          ("balanced on the guess", strat_balanced_guess, 5),
          ("weak buckets first",    strat_weak_first,     5),
          ("least confident",       strat_least_conf,     1),
          ("most confident",        strat_most_conf,      1)]

print(f"{'strategy':24}{'macro-F1':>10}{'spread':>9}   selections")
print("-" * 62)
RESULTS, SDS = {}, {}
for nm, fn, n_seeds in STRATS:
    vals = [score(fn(pool_view, seed=s), nm, show=False) for s in range(n_seeds)]
    RESULTS[nm], SDS[nm] = float(np.mean(vals)), float(np.std(vals))
    print(f"{nm:24}{RESULTS[nm]:>10.3f}{SDS[nm]:>9.3f}   {n_seeds}")

best, worst = max(RESULTS, key=RESULTS.get), min(RESULTS, key=RESULTS.get)
BAR = HOUSE + SPREAD
print("-" * 62)
print(f"\nbest   {best:24} {RESULTS[best]:.3f}")
print(f"worst  {worst:24} {RESULTS[worst]:.3f}")
print(f"spread of {RESULTS[best]-RESULTS[worst]:.3f} macro-F1 for identical money\n")
print(f"the bar to genuinely beat the house was {BAR:.3f}  (its mean plus one spread)")
for k in RESULTS:
    if k != "random (the house)":
        print(f"   {k:24} {'CLEARED IT' if RESULTS[k] > BAR else 'did not clear it'}")

#### ▶ STEP 19 &middot; Why the winner won

In [ ]:
# -------- STEP 19 · Why the winner won --------
rows = []
for nm, fn, _ in STRATS:
    idx, y = buy_labels(fn(pool_view, seed=0))
    rows.append(pd.Series(y).value_counts().reindex(CATS).fillna(0).astype(int).rename(nm))
mix = pd.DataFrame(rows)
mix["lopsided"] = mix[CATS].max(axis=1) - mix[CATS].min(axis=1)
mix["macro_F1"] = [RESULTS[n] for n in mix.index]
mix = mix.sort_values("macro_F1", ascending=False)
print(mix.to_string())

print(f"\ncorrelation between lopsidedness and score: "
      f"{mix.lopsided.corr(mix.macro_F1):.2f}")
print("\nAn even purchase scored well. A lopsided one did not. Nobody was allowed to")
print("balance on the true labels -- that is exactly what we were paying to find out.")

✅ **What just happened, and it is the most useful result of the night.**

Look at the correlation. **The flatter your purchase, the better you did.** Not "the cleverer your
rule", not "the more uncertain the tickets". Just: did you end up with a bit of everything?

And now the genuinely interesting part. **You were never allowed to balance on the true labels** —
those are the thing you are paying for. So how did the winner get an even spread without seeing them?

**It covered the vector space instead.** Cluster the embeddings into 240 groups, buy one ticket from
each. Different categories live in different regions of that space, so spreading across the space gives
you a spread across the categories **as a side effect, for free.**

Why the others lost:

- **most confident** bought tickets the copilot already got right. You paid for answers you had. And
  because confidence tracks "easy bucket", it bought 63 how-to tickets and 8 feature requests.
- **least confident** is real, respected active learning. It lost badly here because at the very start
  you have no decent model to be uncertain *with*, so "uncertain" just means "weird" — ambiguous
  tickets sitting between two categories, hard for a person to label consistently.
- **balanced on the guess** was the right instinct executed on a bad signal. The copilot's guess is
  only 64% accurate, so an even split by guess is an uneven split by truth.
- **weak buckets first** followed our own shopping-list chart and still lost. The advice was not wrong;
  the dose was. Half the budget on three categories starved the other five, and we are scored on all
  eight.

💼 **At work this means:** your selection rule quietly chooses your training distribution, and through
it your model's blind spots. Any rule that ranks by a score will skew that distribution. If you have no
labels yet, **spread across the data, not across a score** — and always look at the mix you ended up
with before you start training.

One more honest note: **random was hard to beat.** Five of six strategies lost to it. When you read a
paper or a vendor deck claiming a clever selection method, ask what plain random sampling scored,
averaged over several draws. Very often nobody checked.

---
# Rungs 6 and 7 · should Cobalt train its own model?

Someone in the room is thinking it, so let us price it honestly.

#### ▶ STEP 20 &middot; RUNGS 6 and 7: should Cobalt train its own?

In [ ]:
# -------- STEP 20 · RUNGS 6 and 7: should Cobalt train its own? --------
LADDER = [
 ("1  use it as it comes",        0,        "minutes",      ZS_F1),
 ("2  rewrite descriptions",      0,        "an hour",      ZS_F1 + 0.00),
 ("4  frozen encoder + head",     BUDGET,   "an afternoon", RESULTS[best]),
 ("5  fine-tune the whole model", 320,      "1-2 weeks",    None),
 ("6  keep pretraining it",       0,        "months",       None),
 ("7  train from scratch",        0,        "$10m and up",  None),
]
print(f"{'rung':30}{'labels':>8}{'cost':>16}   macro-F1")
print("-" * 72)
for nm, n, cost, f1 in LADDER:
    val = f"{f1:.3f}" if f1 is not None else "not worth measuring"
    print(f"{nm:30}{n:>8}{cost:>16}   {val}")

print("\nRoBERTa is the honest answer to 'can we do better than BERT'.")
print("Same architecture. Same masked-language-model idea. They just trained it")
print("longer, on more text, with better choices. No new invention required --")
print("and it still cost millions of dollars of compute.")

✅ **What just happened.** We put the whole ladder in one table, and the shape of the answer is clear.

**Rung 6, continued pretraining**, is real and occasionally right: keep playing fill-in-the-blank, but
on *your* documents, so the model finally learns what "Okta" and "SAML" mean in your world. Worth it
if you have millions of words of genuinely unusual text — legal filings, clinical notes, chip design
docs. Cobalt has support tickets that are mostly ordinary English. **Not worth it.**

**Rung 7, training from scratch.** The reason **RoBERTa** matters is that it is the cleanest possible
demonstration of how little you gain and how much you pay. It is BERT's architecture, unchanged, with
a better training recipe. That is what millions of dollars of compute buys at the frontier: a better
recipe for the same idea. You are not going to beat that with a support-ticket dataset.

💼 **At work this means:** "should we train our own model?" is nearly always "no", and the useful
follow-up is "what is the highest rung we can reach with the data we already have?" For Cobalt tonight
that was rung 4, and it cost one afternoon.

#### ▶ STEP 21 &middot; The whole ladder, with your numbers

In [ ]:
# -------- STEP 21 · The whole ladder, with your numbers --------
# labelling the ENTIRE pool, so we can see what all that extra work would have bought
full = f1_score(y_test, LogisticRegression(max_iter=2000, C=4.0)
                .fit(X_pool, y_pool).predict(X_test), average="macro")

bars = [(f"no labels\n(Lecture 13)",        ZS_F1,          "#0EA5E9"),
        (f"{BUDGET} labels\nchosen badly",  RESULTS[worst], "#E11D48"),
        (f"{BUDGET} labels\nat random",     HOUSE,          "#94A3B8"),
        (f"{BUDGET} labels\nchosen well",   RESULTS[best],  "#059669"),
        (f"all {len(pool):,} labels",        full,           "#7C3AED")]

fig, ax = plt.subplots(figsize=(9.4, 4.0))
bb = ax.bar([b[0] for b in bars], [b[1] for b in bars],
            color=[b[2] for b in bars], width=.62)
for b, v in zip(bb, [b[1] for b in bars]):
    ax.text(b.get_x()+b.get_width()/2, v+.012, f"{v:.3f}", ha="center",
            fontsize=11.5, fontweight="bold")
ax.set_ylabel("macro-F1"); ax.set_ylim(0, max(b[1] for b in bars)+.12)
ax.set_title("Everything we did tonight, on one axis")
ax.grid(axis="y", alpha=.25); ax.set_axisbelow(True)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print(f"{BUDGET} well-chosen labels got {RESULTS[best]/full:.0%} of the way to "
      f"labelling all {len(pool):,}.")
print(f"The other {len(pool)-BUDGET:,} tickets, roughly five times the work, bought the rest.")

---
## The one idea to take with you

Last lecture: *someone else already paid to teach a model to read; point it at your problem.*

Tonight:

> **Adapting a model is mostly a purchasing decision, not an engineering one.** How many labels, which
> ones, and which rung of the ladder — those are the choices that move the number. The expensive rung
> is not the best rung, and 240 tickets chosen carefully beat 240 chosen cleverly.

### What you can do at work on Monday
1. Plot the learning curve before committing to a labelling project. Find where it flattens.
2. Try the cheap rung first: frozen embeddings and a simple classifier. Make the expensive option
   prove it is better.
3. Check the class mix of whatever your selection rule produced. Look for the zeros.
4. When someone proposes training a model from scratch, ask what rung they have already tried.